CASO PRATICO: SENTIMENT ANALYSIS VIA API REST

Costruiamo un servizio di sentiment analysis.
immagina di dover analizzare migliaia di recensioni di clienti in tempo reale.
Non serve solo un modello accurato, ma un sistema che sia veloce, testabile e che non sprechi risorse.

Simulazione di un progetto aziendale.
- Sviluppo End-to-End: integrare DistlBERT in su server FastAPI asincrono
- Testing e Analisi: collaudiamo il modello, micurare la latenza di inferenza tramite cURL e Postman, misuriamo quanto tempo di mette a rispondere, anche sotto stress.
- Orchestrazione: configurare Docker Compose con Redis per il caching delle risposte. Per affiancare alla nostra API un db Redis che funge da memoria a brevissimo termine per evitare calcolo inutili

Esempio
immagina questo testo JSON: {"text": "il prodotto è eccellente}
il servizio restituisce una risposta strutturata JSON: {"label": "positive", "score": 0.98}
Ogni componente del servizio ho uno scopo preciso:
* Cliente, Postman o programma ERP: invia la richiesta HTTP
* FastAPI: espone l'indirizzo del servizio e gestisce le richieste
* Pydantic: controlla la struttura e vincoli dei dati
* Tokenizer: trasforma il testo in numeri utilizzabili dal modello
* DistlBERT specializzato: calcola i punteggi delle classi
* Redis: conserve tamporaneamente risultati già calcolati
* Docker Compose: configura ed avvia i servizi insieme
FastAPI non analizza il sentimento, Redis non analizza il sentimento. Il modello analizza il sentimento. Gli altri componenti rendeono utilizzabile quel calcolo.

La Sentiment Analysis classifica l'orientamento espresso in un testo.

Integrazione del Modello
DistlBERT incontra FastAPI
Il nostro protagonista è DistlBERT, il fratello minore e più agile di BERT. E' perfetto per questo compito perchè mantiene circa il 97% delle performance dell'originale ma è molto più leggero e veloce in inferenza.

DistlBERT non è automaticamente un classificatore di sentiment. DistlBERT è un'architettura di modello. Per questo esercizio serve una sua versione già specializzata nel riconoscimento del sentimento, esempio "distilbert/distilbert-base-uncased-finetuned-sst-2-english".
Questa versione è addestrata in inglese e classifica positivo/negativo. non offre, automaticamente, una classe neutra, e non è un modello da scegliere per classificare mail italiane o multilingua.
Se la richiesta è neutra, esempio "please send the order confirmation", un classificatore binario come quello scelto, deve comunque scegliere tra positivo o negativo.
Un modello restituisce le classi che conosce, anche quando l'esigenza richiederebbe altre categorie.
Pertanto, se la necessità è classificare la mail sarà necessario distinguere tra diversi casi:
- il cliente esprime insoddisfazione?           -   sentiment
- il cliente invia un ordine o un reclamo       -   classificazione dell'intento
- al cliente serve un intervento entro oggi?    -   urgenza
- quali articoli e quantita richiede?           -   estrazione strutturata
Una mail può essere cortese, quindi apparentemente 'positiva' ma allo stesso tempo contenere un fermo produzione urgente.

Chiamarlo in FastAPI non significa solo chiamare un funzione, significa gestire il ciclo di vita dei pesi.
Vediamo come caricare un modello 'Transformers' all'interno di un'architettura web moderna, assicurandoci che il caricamento dei pesi avvenga una sola volta all'avvio, come un motore che viene lasciato acceso al minimo e pronto a scattare non appena arriva una richiesta HTTP.
La Pipeline di Hugging Face la utilizziamo per nascondere la complessità della tokenizzazione e l'inferenza esponendo il tutto tramite un endpoint POST valido.

Come si presenta l'architettura interna di questo server?
Architettura del Server AI
Dal testo grezzo alla classe predetta.
Quando una recensione arriva all'endpoint predict, parte prima il filtro di pydantic, se il testo non è valido, lo fermiamo subito. Una volta validato il dato entra nel nostro mondo asincrono. 
Il server sta risolvendo un probelma di classificazione. Siccome il calocolo è pesante lo trasferiamo in un thread pool dedicato per non paralizzare l'intero sever mentre la GPU macina i numeri.

- Lifespan Management: caricamento del modello DistlBERT nello stato globale dell'app per massimizzare il riutilizzo della memoria.
- Schema Pydantic: definizione di 'SentimentRequest' per validare l'input dell'utente esempio: una stringa non vuota.
- Async Inference: esecuzione della funzione di predizione in un thread pool per non bloccare l'event loop di FastAPI durante i calcoli della CPU
Il server riceve un documento 'd' e restituisce una distribuzione di probabilità sulla polarità del testo

Vediamo nel dettaglio come il modello sia sempre acessibile.
Dettagli di Implementazione
Per rendere il sistema robusto, implementiamo il modello come un singleton. Immaginalo come una unica risorsa condivisa che vive nello stato dell'applicazione.

Si leggo i pesi del modello e li si mettono nella memoria del computer o della GPU.
E' sbagliato farlo dentro ogni richiesta, ma va avviato all'avvio dell'applicazione
FastAPI mette a disposizione il meccanismo 'lifespan' per inizializzare risorse all'avvio e rilasciarle alla chiusura.

Usiamo l'astrazione della pipeline perchè sa come gestire il padding delle frasi e il troncamento se un utente manda un libro intero invece di una recensione. 
Infine definiamo un modello di riposta chiaro, non vogliamo che il client riceva dati confusi ma un json pulito, con un label come positivo ed un punteggio di confidenza tra 0 e 1.

- Model Singleton: garantire che l'istanza del modello sia accessibile globalmente tramite le dipendenze di FastAPI o l'oggetto app.state.
- Pipeline Abstraction: verifica che il server restituisca errori di validazione Pydantic (422 Unprocessable Entity) per input malformati.
- Response Modeling: test di invio simultaneo di più richieste per osservare come il thread pool gestisce l'accodamento dell'inferenza CPU-bound

Ma quanto tempo ci mette questo processo a completarsi
La latenza è il nostro nemico numero 1
In un Transofrmer il tempo di esecuzione non è fisso, cresce linearmente con la lunghezza del testo che inviamo.
Ogni parola aggiunge un carico nella matrici di attenzione
Oltre al calcolo puro c'è un piccolo overhead di gestione web. Fare il profiling significa capire quanto di quel tempo è speso nei neuroni del modello e quanto nei cavi della rete.
Questo ci permette di decidere se il nostro hardware è sufficiente o se dobbiamo ottimizzare ulteriormente.

Ora che il nostro server è in piedi, dobbiamo imparare ad interrogarlo come dei professionisti.
Testing e Monitoraggio
Verifica del servizio REST
Un'API AI deve essere testata rigorosamente non solo per la correttezza del risultato, ma anche per la stabilità delle prestazioni sotto carico.
Strumenti standard come Postman e cURL ci permettono di simulare chiamate reali e analizzare gli header di risposta e i tempi di esecuzione.
Verifichiamo che il server risponda correttamente anche sotto carico e che i tempi di esecuzione siano stabili nel tempo.

Come analizzare queste risposte dal terminale.
Analisi della Risposta
Validare il contratto API
- cURL Testing: utilizzato per test rapidi, invio di JSON strutturati tramite riga di comando per verificare la corretta deserializzazione lato server.
- Postman Collections: facciamo un passo avanti rispetto cURL, creiamo delle collezioni di test, organizzazione dei test cases (test positivi, negativi, sarcastici) per la regressione automatizzata
- Time to First Byte (TTFB): monitoriamo la varianza dei tempi di risposta. Se la varianza è alta vuol dire che il server non è stabile
La stabilità del servizio si misura tramite il monitoraggio della varianza dei tempi di risposta sulle chiamate sequenziali

Cosa succede però quando le cose vanno male
La gestione degli errori è ciò che distingue uno script da un prodotto, cosa succede se un utente manda una stringa vuota? o se manda un testo da 10.000 parole eccedendo il limite di distlBERT?
Dobbiamo catturare queste eccezione e restituire codici di errori significativi come 422 per dati malformati o 400 per input troppo lunghi
- Input troppo lungo: simulazione di invio di testi che superano i 512 token e gestione dell'errore tramite eccezioni HTTP 400 personalizzate
- Empty Payloads: verifica che il server restituisca errori di validazione Pydantic (422 Unprocessable Entity) per input malformati
- Concurrency Stress: test di invio simultaneo di più richieste per osservare come il thread pool gestisce l'accodamento dell'inferenza CPU-bound
Dobbiamo conoscere il punto di rottura prima dei nostri utenti

C'è però un costo nascosto nel viaggiare su protocollo REST, ogni volta che inviamo un pacchetto JSON paghiamo un dazio in termini di tempo.
In un'API REST, il tempo di rete e la serializzazione JSON aggiungono un ritardo che dobbiamo minimizzare per servizi real-time.
Dobbiamo valutare il tempo della rete rispetto al tempo del calcolo utile del modello.
Se il modello è velocissimo ma il JSON è enorme, la rete è il collo di bottiglia.
Dobbiamo minimizzare il rapporto tra tempo del modello e tempo di rete, rendendo il passaggio dei dati il più snello possibile, per mantenere l'interattività dell'applicazione.

Per risolvere il problema della velocità dobbiamo orchestrare.

Orchestrazione e Caching
Docker Compose e Redis
il servizio va scalato aggiungendo uno stato di memoria persitente per evitare di ricalcolare sentimenti per frasi già viste.
Utiliziamo Docker Compose per orchestrare i due container: uno per l'API FastAPI e uno per Redis, il database in-memory per il caching
Se l'utente manda la frase: "ottimo prodotto" è stupido far lavoare la CPU 100 volte per lo stesso calcolo, entra in gioco Redis, un database in memory ultra veloce. Usiamo Docket Compose per far nascere un ecosistema, un container per il cervello (il nostro modello) ed uno per la memoria a breve termine, Redis.
Il servizio cerca, come prima cosa, il testo in Redis, se lo trova restituisce la predizione fatta in precedenza; se non lo trova esegue l'inferenza e salva il risultata in redis.
Per fare il confronto tra la frase appena inviata dal cliente, e le frasi prensenti in Redis (frasi precedenti)
E' come appuntarsi le risposte già date.
Questra sinnergia trasforma le performance.

Sinergia dei Servizi
Scalabilità e risparmio computazionale.
Quando arriva un testo, ne calcoliamo l'impronta digitale (quindi l'hash) e se è presente in Redis (cache hit) restituiamo la risposta subito, se non c'è (cache miss) mandiamo tutto al modello per l'inferenza, attendiamo la risposta e mandiamo tutto in Redis per la prossima volta.
- Redis Cache: memorizzazione dei risultati dell'inferenza usanto l'hash del testo di input come chiave univoca.
- Docker Compose YAML: definizione dei servizi, delle reti interne e dei volumi necessari per far comunicare API e Cache.
- Cache Hit vs Miss: logica per cui se il risultato esiste in Redis viene restituito in microsecondi, altrimenti si procede con l'inferenza DistlBERT.
Il caching riduce drasticamente il carico sulla CPU/GPU per query ripetitive, migliorando la scalabilità orizzontale.

Tutto questo è orchestrato con poche righe di configurazione Docker Compose

Con Docker Compose si permette di definire le dipendenze
- Depends_on: istruzione per garantire che il container Redis sia attivo e pronto prima dell'avvio dell'applicazione FastAPI
- Environment Vars: usiamo variabili d'ambiente per dire al codice python dove trovare Redis (passaggio delle credenziali e dell'host di Redis al container Python)
- Volume Persistence: usare volumi docker per le cartelle dei modelli di Hugging face in modo che se eliminate il container non si deve riscaricare i 500 mega per i pesi del modello 

Ma quanta probabilità abbiamo da essere così fortunati da trovare un dato già pronto?
E' statistica, il tempo medio di risposta dipende dalla probabilita p che gli utenti facciano domande ripetitive. Se gestisci commenti sociale p sarà altissima, se gestisci testi unici p sarà bassa.
Tuttavia, con un p basso, Redig protegge il sistema dagli attacchi di tipo replay dove un maleintenzionato vuole saturare la CPU inviando la stessa richiesta infinite volte.

Il servizio va valutato non solo per la qualità del modello (classifica correttamente i testi inviati?) ma anche per la qualità del servizio (risponde con tempi e con affidabilità accettabili?)
Un sistema velocissimo che classifica male, non è un buon sistema.

per l'esercizio vedi cartella:
.\Epicode_Python_AI_MachineLearning\PYTHON\Modulo_6_Generative\5_API\12_CasoPratico_Sentiment